In [1]:
# Biblioteki
import math
import random
import numpy as np

In [22]:
# Zmienne globalne
LICZ_WIER = 6
LICZ_KOL = 7

# Głębokość algorytmu alpha-beta
GLEBOKOSC = 6 # głębokość drzewa 
GRACZ = 1   # gracz ma żeton o indeksie 1
BOT = 2     # bot ma żeton o indeksie 2

In [61]:
# Podstawowe funkcje
def tworz_plansze():
    plansza = np.zeros((LICZ_WIER, LICZ_KOL))
    return plansza.astype(int)

def rysuj_plansze(plansza):
    '''Rysujemy bardziej czytelną planszę'''
    for wier in range(LICZ_WIER):
        print("|", end="")
        for kol in range(LICZ_KOL):
            if plansza[wier][kol] == 0:
                print("   |", end="")
            if plansza[wier][kol] == 1: # 1 dla gracza X
                print(" X |", end="")

            if plansza[wier][kol] == 2: # 2 dla gracza O
                print(" O |", end="")
        print()     

    print("-----------------------------")
    print("|", end="")       
    for k in range(LICZ_KOL):
        print(f" {k+1} |", end="")
    print()
    print()

def czy_ruch_dozwolony(plansza, kolumna):
    '''Sprawdzamy, czy ruch w tej kolumnie jest dozwolony, tzn. czy kolumna nie jest już zapełniona'''
    for wiersz in range(LICZ_WIER):
        if plansza[wiersz][kolumna] == 0:
            return True
    return False

def znajdz_wiersz(plansza, kolumna):
    '''Szukamy w danej kolumnie  pierwszego wiersza od dołu, do którego możemy wstawić żeton i zwracamy, który to wiersz'''
    for wiersz in range(LICZ_WIER-1, -1, -1):
        if plansza[wiersz][kolumna] == 0:
            return wiersz

def wstaw_zeton(plansza, kolumna, zeton):
    '''Wstawiamy żeton w dane miejsce na planszy, uwzględniając "grawitację" '''
    wiersz  = znajdz_wiersz(plansza, kolumna)
    #print(f"wiersz: {wiersz}")
    plansza[wiersz][kolumna] = zeton
    return plansza

def mozliwe_ruchy(plansza):
    '''Zwraca listę kolumn, do których możemy wrzucić żetony'''
    lista_ruchow = list()

    for kolumna in range(LICZ_KOL):
        if czy_ruch_dozwolony(plansza, kolumna) == True:
            lista_ruchow.append(kolumna)
    return lista_ruchow

def czy_plansza_pelna(plansza):
    return len(mozliwe_ruchy(plansza)) == 0 # brak ruchów do wykonania (mamy remis)

def czy_ruch_wygrywajacy(plansza, zeton):
    '''Sprawdzamy, czy dany ruch spowoduje wygraną. Sprawdzamy poziomy, piony, skosy'''
    
    # Musimy sprawdzić czy w jednym wierszu są obok siebie 4 takie same żetony
    # Poziom
    for wier in range(LICZ_WIER):
        for kol in range(LICZ_KOL-3): # Sprawdzamy od lewej strony, dlatego nie musimy sprawdzać do końca
            if plansza[wier][kol] == zeton and plansza[wier][kol+1] == zeton and plansza[wier][kol+2] == zeton and plansza[wier][kol+3] == zeton:
                return True
    # Pion
    for wier in range(LICZ_WIER-3): # Sprawdzamy od góry, dlatego nie musimy sprawdzać do końca
        for kol in range(LICZ_KOL):
            if plansza[wier][kol] == zeton and plansza[wier+1][kol] == zeton and plansza[wier+2][kol] == zeton and plansza[wier+3][kol] == zeton:
                return True

    # Skosy (lewy górny róg)
    for wier in range(LICZ_WIER-3): # Sprawdzamy od góry, dlatego nie musimy sprawdzać do końca. Będziemy sprawdzać w dół i w prawo
        for kol in range(LICZ_KOL-3):
            if plansza[wier][kol] == zeton and plansza[wier+1][kol+1] == zeton and plansza[wier+2][kol+2] == zeton and plansza[wier+3][kol+3] == zeton:
                return True

    # Skosy (prawy górny róg)
    for wier in range(LICZ_WIER-3): # Będziemy sprawdzać w dół i w lewo 
        for kol in range(3, LICZ_KOL):
            if plansza[wier][kol] == zeton and plansza[wier+1][kol-1] == zeton and plansza[wier+2][kol-2] == zeton and plansza[wier+3][kol-3] == zeton:
                return True
            
    # Jeśli gra się jeszcze nie skończy
    return False

# Podstawowa heurystyka
def heurystyka_1(plansza, zeton):
    '''Funkcja do obliczania wartości pozycji. Każda pozycja na planszy ma swoją ustaloną wartość'''

    wynik = 0 # początkowy wynik
    
    if zeton == 1: # jeśli ruch gracza 1
        zeton_gracza = 1
        zeton_przeciwnika = 2
    else: # jeśli ruch gracza 2
        zeton_gracza = 2
        zeton_przeciwnika = 1 

    # Faworyzujemy środek planszy - mamy tam najwięcej możliwości na zdobycie wygrywającej 4
    tabela_wartosci = np.array([[3, 4, 5, 7, 5, 4, 3],
                                [4, 6, 8, 10, 8, 6, 4],
                                [5, 7, 11, 13, 11, 7, 5],
                                [5, 7, 11, 13, 11, 7, 5],
                                [4, 6, 8, 10, 8, 6, 4],
                                [3, 4, 5, 7, 5, 4, 3]])
    
    # Wcześniejsza wersja
    # tabela_wartosci = np.array([[2, 4, 6, 8, 6, 4, 2],
    #                             [3, 6, 9, 12, 9, 6, 3],
    #                             [4, 8, 12, 16, 12, 8, 4],
    #                             [4, 8, 12, 16, 12, 8, 4],
    #                             [3, 6, 9, 12, 9, 6, 3],
    #                             [2, 4, 6, 8, 6, 4, 2]])

    
    wynik_gracza = np.sum(tabela_wartosci[plansza==zeton_gracza]) # sumujemy te wartości, gdzie jest żeton gracza
    wynik_przeciwnika = np.sum(tabela_wartosci[plansza==zeton_przeciwnika]) # sumujemy te wartości, gdzie jest żeton przeciwnika

    wynik = wynik_gracza - wynik_przeciwnika # ostateczny wynik ruchu
    return wynik
    


In [ ]:
# Algorytm minmax

def minmax(plansza, glebokosc, alpha, beta, gracz_max):

    # Sprawdzamy czy dla tego rozstawienia żetonów gra się kończy
    if czy_ruch_wygrywajacy(plansza, BOT):
        # glebokosc - preferujemy szybkie wygrane
        return 1000000000 - 3*glebokosc  # Wygrał bot - gracz max
    elif czy_ruch_wygrywajacy(plansza, GRACZ):
        return -1000000000 + 3*glebokosc # Wygrał człowiek - gracz min
    elif czy_plansza_pelna(plansza):
        return 0   # remis 

    # jeśli dotrzemy do liści drzewa ("zejdziemy" na zadaną głębokość) zaczynamy oceniać pozycje
    if glebokosc == 0:
        return(heurystyka_1(plansza,BOT))

    lista_kolumn = mozliwe_ruchy(plansza)

    # Losujemy kolejność sprawdzania kolumn, aby bot był mniej powtarzalny
    #random.shuffle(lista_kolumn)

    if gracz_max == True: # Ruch Bota (MAX)
        max_ocena = -np.inf
        for kolumna in lista_kolumn:
            tymczasowa_plansza = plansza.copy() # sprawdzamy ruchy na kopiach planszy, aby nie modyfikować planszy na której gramy
            wstaw_zeton(tymczasowa_plansza, kolumna, BOT)

            # Oceniamy ruch gracza MIN
            ocena = minmax(tymczasowa_plansza, glebokosc-1, alpha, beta, False) # bo teraz ruch wykonuje gracz min
            max_ocena = max(max_ocena, ocena)

            alpha = max(alpha, ocena)
            if beta <= alpha:
                break

        return max_ocena
    
    else:
        # Ruch MIN
        min_ocena = np.inf
        for kolumna in lista_kolumn:
            tymczasowa_plansza = plansza.copy() # sprawdzamy ruchy na kopiach planszy, aby nie modyfikować planszy na której gramy
            wstaw_zeton(tymczasowa_plansza, kolumna, GRACZ)

            # Teraz oceanimy ruch MAX
            ocena = minmax(tymczasowa_plansza, glebokosc-1, alpha, beta, True) # bo teraz ruch wykonuje gracz max
            min_ocena = min(min_ocena, ocena)
            
            beta = min(beta, ocena)
            if beta <= alpha:
                break

        return min_ocena


In [63]:
def najlepszy_ruch_bota(plansza):
    dostepne_ruchy = mozliwe_ruchy(plansza)

    # Sprawdzamy czy jest ruch wygrywający
    for kol in dostepne_ruchy:
        tymczasowa_plansza = plansza.copy()
        wstaw_zeton(tymczasowa_plansza, kol, BOT)
        if czy_ruch_wygrywajacy(tymczasowa_plansza, BOT):
            #print(f"Natychmiastowa wygrana w kolumnie {kol+1}")
            return kol

    # Sprawdzamy czy musimy zablokować Gracza
    for kol in dostepne_ruchy:
        tymczasowa_plansza = plansza.copy()
        wstaw_zeton(tymczasowa_plansza, kol, GRACZ)
        if czy_ruch_wygrywajacy(tymczasowa_plansza, GRACZ):
            #print("Bot blokuje wygraną Gracza")
            return kol

    najlepszy_wynik = -np.inf
    najlepsza_kolumna = dostepne_ruchy[0] #pierwszy dostępny ruch

    # Sprawdzamy wszystkie możliwe ruchy
    for kolumna in mozliwe_ruchy(plansza):
        tymczasowa_plansza = plansza.copy() # sprawdzamy na kopiach planszy
        wstaw_zeton(tymczasowa_plansza, kolumna, BOT)

        # Wywołujemy minmax dla gracza min (bo Bot już wstawił swój żeton)
        wynik = minmax(tymczasowa_plansza, GLEBOKOSC, -np.inf, np.inf, False)
        # Porównanie wyników
        if wynik > najlepszy_wynik:
            najlepszy_wynik = wynik
            najlepsza_kolumna = kolumna

    return najlepsza_kolumna

In [64]:
def graj():
    plansza = tworz_plansze()
    koniec_gry = False
    # tura 0 - Gracz, 1 - Bot

    # Wybór kto zaczyna
    print("Kto zaczyna?")
    print("1. Gracz(X)")
    print("2. Bot(O)")

    kto_pierwszy = input("Wybierz kto zaczyna (1 lub 2):")

    if kto_pierwszy == "1":
        tura = 0
        print("Zaczyna Gracz(X)")
    else:
        tura = 1
        print("Zaczyna Bot(O)")

    rysuj_plansze(plansza)

    while not koniec_gry:
        # Ruch Gracza
        if tura == 0:
            wybor = input("Wybierz kolumnę (1-7): ")
            
            
            # Obsługa wyjątków
            try:
                kol = int(wybor)-1
            except ValueError:
                print("Podaj liczbę")
                continue
            
            if not 1 <= int(wybor) <= LICZ_KOL:
                print("Ruch niedozwolony")
                continue
            
            if kol in mozliwe_ruchy(plansza):
                wstaw_zeton(plansza, kol, GRACZ)
                if czy_ruch_wygrywajacy(plansza, GRACZ):
                    rysuj_plansze(plansza)
                    print("Wygrał gracz")
                    koniec_gry = True
                # Gracz wykonał ruch, kolej bota
                tura = 1
            else:
                print("Ruch niedozwolony")

        # Ruch Bota
        else:
            print("Ruch Bota...")
            kol = najlepszy_ruch_bota(plansza)
            wstaw_zeton(plansza,kol, BOT)
            print(heurystyka_1(plansza,BOT))
            print("Bot wykonał ruch. Kolej na Gracza...")

            if czy_ruch_wygrywajacy(plansza, BOT):
                rysuj_plansze(plansza)
                print("Wygrał Bot")
                koniec_gry = True
            # Bot wykonał ruch, pora na gracza
            tura = 0

        rysuj_plansze(plansza)

        if not koniec_gry and czy_plansza_pelna(plansza):
            print("Remis")
            koniec_gry = True
            
graj()


Kto zaczyna?
1. Gracz(X)
2. Bot(O)
Zaczyna Gracz(X)
|   |   |   |   |   |   |   |
|   |   |   |   |   |   |   |
|   |   |   |   |   |   |   |
|   |   |   |   |   |   |   |
|   |   |   |   |   |   |   |
|   |   |   |   |   |   |   |
-----------------------------
| 1 | 2 | 3 | 4 | 5 | 6 | 7 |

|   |   |   |   |   |   |   |
|   |   |   |   |   |   |   |
|   |   |   |   |   |   |   |
|   |   |   |   |   |   |   |
|   |   |   |   |   |   |   |
|   |   |   | X |   |   |   |
-----------------------------
| 1 | 2 | 3 | 4 | 5 | 6 | 7 |

Ruch Bota...
3
Bot wykonał ruch. Kolej na Gracza...
|   |   |   |   |   |   |   |
|   |   |   |   |   |   |   |
|   |   |   |   |   |   |   |
|   |   |   |   |   |   |   |
|   |   |   | O |   |   |   |
|   |   |   | X |   |   |   |
-----------------------------
| 1 | 2 | 3 | 4 | 5 | 6 | 7 |

|   |   |   |   |   |   |   |
|   |   |   |   |   |   |   |
|   |   |   |   |   |   |   |
|   |   |   | X |   |   |   |
|   |   |   | O |   |   |   |
|   |   |   | X |   |  